In [1]:
from _src.Utils.CompositionLoader import CompositionExcelLoader
from _src.Composition.CompositionV2 import Composition
from _src.PhaseStability.TwoPhaseStabilityTestV3 import TwoPhaseStabilityTest
from _src.VLE.PhaseEquilibriumNewtonV2 import PhaseEquilibriumNewton
from _src.Utils.FluidPropertiesCalculatorV2 import FluidPropertiesCalculator
from _src.EOS.BaseEOS import EOSType

## Загрузка состава из БД

In [2]:
composition_db = Composition.from_db(r'/Users/daniilsidorenko/Desktop/pvt_module/PVTcalc/models.json')

In [3]:
comp_KRSNL = composition_db.KRSNL_PVTSIM

In [4]:
comp_KRSNL.normalize_composition()

In [ ]:
comp_KRSNL.composition_data.keys()

In [39]:
comp_PRRZLM = composition_db.PRRZLM_MDT_TEST


## Старая загрузка составов

In [ ]:
xl_loader = CompositionExcelLoader(r'C:\Users\user\Desktop\PVT_TSU\diss\prrzlm_mdt.xlsx')
comp_dict = xl_loader.load(header=True, sheet = 'to_model')

In [ ]:
composition = Composition(comp_dict, eos_name= EOSType.PREOS)
composition.evaluate_composition_data({'critical_temperature': 'kesler lee',
                                                    'critical_pressure': 'kesler lee',
                                                    'acentric_factor': 'riazi al sahhaf',
                                                    'critical_volume': 'hall yarborough',
                                                    'Kw': 'k watson',
                                                    'shift_parameter': 'jhaveri youngren'})

composition.edit_component_properties('C36', 'molar_mass', 650)

In [ ]:
composition2 = Composition(comp_dict)
composition2.evaluate_composition_data({'critical_temperature': 'pedersen',
                                                    'critical_pressure': 'pedersen',
                                                    'acentric_factor': 'riazi al sahhaf',
                                                    'critical_volume': 'hall yarborough',
                                                    'Kw': 'k watson',
                                                    'shift_parameter': 'jhaveri youngren'})

In [ ]:
composition2.normalize_composition()

In [ ]:
composition.composition_data.keys()

In [ ]:
for i in composition.composition_data['acentric_factor'].values():
    print(i)

## Расчет флеша

In [51]:
from _src.VLE.Flash import Flash
from _src.Utils.Conditions import Conditions

In [52]:
conds_stable = Conditions(185, 110)
conds_unstable = Conditions(151, 110)

In [53]:
flash_obj1 = Flash(comp_KRSNL, conds_stable)
flash_obj2 = Flash(comp_KRSNL, conds_unstable)

In [54]:
flash_res1 = flash_obj1.calculate()

In [ ]:
flash_res1

In [ ]:
flash_res2

In [5]:
from _src.Experiments.DLE2 import DLE

In [6]:
dle_obj = DLE(comp_KRSNL, [250, 200, 150, 100, 50, 30, 10, 5], 300, 110)

In [7]:
dle_data = dle_obj.calculate()

2026-06-14 20:52:56,446 | _src.PhaseDiagram.new_methodv2 | INFO | Инициализация: T=383.14 K, P_min=0.0100, P_max=1000.0000
2026-06-14 20:52:56,448 | _src.PhaseDiagram.new_methodv2 | INFO | Запуск поиска давления насыщения
2026-06-14 20:52:56,448 | _src.PhaseDiagram.new_methodv2 | INFO | 🔍 Поиск bracket: start=[0.010, 1000.000]
2026-06-14 20:52:56,621 | _src.PhaseDiagram.new_methodv2 | INFO | Диапазон поиска: unstable=184.5785 bar -> stable=185.5550 bar
2026-06-14 20:52:56,789 | _src.PhaseDiagram.new_methodv2 | INFO | Интервал схлопнулся на итерации 12. Возврат P=185.0415 bar


In [8]:
dle_data

[FlashResult(pressure=300, temperature=383.14, vapor=PhaseState(mole_fraction=0.0, composition={'N2': 0.0032799672003279963, 'CO2': 0.014229857701422984, 'C1': 0.3934560654393456, 'C2': 0.10477895221047788, 'C3': 0.08000919990800091, 'iC4': 0.010309896901030988, 'nC4': 0.033139668603313965, 'iC5': 0.00961990380096199, 'nC5': 0.01888981110188898, 'C6': 0.023539764602353977, 'C7': 0.03264967350326496, 'C8': 0.03346966530334696, 'C9': 0.027859721402785968, 'C10': 0.023979760202397976, 'C11': 0.01967980320196798, 'C12': 0.01625983740162598, 'C13': 0.016019839801601984, 'C14': 0.014179858201417985, 'C15': 0.011739882601173986, 'C16': 0.010029899701002988, 'C17': 0.00868991310086899, 'C18': 0.00812991870081299, 'C19': 0.007559924400755992, 'C20': 0.0066999330006699935, 'C21': 0.0052699473005269944, 'C22': 0.0051299487005129945, 'C23': 0.004369956300436995, 'C24': 0.003959960400395996, 'C25': 0.004059959400405996, 'C26': 0.0032799672003279963, 'C27': 0.003149968500314997, 'C28': 0.00265997340

In [9]:
dle_obj._dle_df

,pressure,temperature,is_two_phase,vapor_mole_frac,liquid_mole_frac,vapor_composition,liquid_composition,vapor_density,liquid_density,vapor_molar_density,...,vapor_molar_volume,liquid_molar_volume,vapor_molecular_ weight,liquid_molecular_ weight,vapor_viscosity,liquid_viscosity,vapor_z,liquid_z,Bo,Rs
0,300.000000,383.14,False,0.000000,1.000000,"{'N2': 0.0032799672003279963, 'CO2': 0.0142298...","{'N2': 0.0032799672003279963, 'CO2': 0.0142298...",NaN,0.652477,NaN,...,NaN,139.016422,NaN,90.705065,NaN,0.050051,NaN,1.309872,1.484591,159.942995
1,185.041525,383.14,True,0.000024,0.999976,"{'N2': 0.00918689651221849, 'CO2': 0.016795311...","{'N2': 0.003279827123635907, 'CO2': 0.01422979...",0.169020,0.630973,0.007054,...,141.758225,143.756811,23.959928,90.706648,0.022835,0.048831,0.823870,0.835486,1.535179,159.942995
2,250.000000,383.14,False,0.000000,1.000000,"{'N2': 0.003279827123635907, 'CO2': 0.01422979...","{'N2': 0.003279827123635907, 'CO2': 0.01422979...",NaN,0.644045,NaN,...,NaN,140.839069,NaN,90.706648,NaN,0.049568,NaN,1.105871,1.504020,159.942995
3,200.000000,383.14,False,0.000000,1.000000,"{'N2': 0.003279827123635907, 'CO2': 0.01422979...","{'N2': 0.003279827123635907, 'CO2': 0.01422979...",NaN,0.633155,NaN,...,NaN,143.261369,NaN,90.706648,NaN,0.048953,NaN,0.899913,1.529888,159.942995
4,150.000000,383.14,True,0.147693,0.852307,"{'N2': 0.008416400137427027, 'CO2': 0.01764797...","{'N2': 0.0023897288874798472, 'CO2': 0.0136374...",0.133894,0.651940,0.005697,...,175.522002,156.996745,23.501359,102.352410,0.021233,0.050183,0.826922,0.739645,1.428950,122.474995
5,100.000000,383.14,True,0.204004,0.795996,"{'N2': 0.006802202857539472, 'CO2': 0.01966333...","{'N2': 0.00125886724940667, 'CO2': 0.012093118...",0.086837,0.681023,0.003697,...,270.461358,179.971765,23.486080,122.564860,0.019241,0.051803,0.849467,0.565257,1.303893,78.365272
6,50.000000,383.14,True,0.220481,0.779519,"{'N2': 0.0043201125184106955, 'CO2': 0.0237594...","{'N2': 0.00039301727645798477, 'CO2': 0.008793...",0.044237,0.710628,0.001758,...,568.832917,211.241353,25.163515,150.114090,0.017581,0.052749,0.893298,0.331734,1.193007,40.418203
7,30.000000,383.14,True,0.105854,0.894146,"{'N2': 0.00254437561534141, 'CO2': 0.027458610...","{'N2': 0.00013832650157082847, 'CO2': 0.006583...",0.028596,0.722947,0.001029,...,971.769072,227.673331,27.788392,164.595744,0.016935,0.052729,0.915642,0.214524,1.149700,26.216434
8,10.000000,383.14,True,0.156114,0.843886,"{'N2': 0.0008079286861857894, 'CO2': 0.0287554...","{'N2': 1.4453920442613415e-05, 'CO2': 0.002482...",0.012164,0.740002,0.000332,...,3016.281534,254.401790,36.688663,188.257824,0.015953,0.052113,0.947356,0.079903,1.084118,7.488752
9,5.000000,383.14,True,0.067528,0.932472,"{'N2': 0.00019065006196509462, 'CO2': 0.022860...","{'N2': 1.6941663202945799e-06, 'CO2': 0.001006...",0.007550,0.746446,0.000164,...,6099.208022,266.002063,46.049064,198.556283,0.015325,0.051684,0.957822,0.041773,1.057005,0.652665


In [10]:
from _src.Experiments.SeparatorTest2 import SeparatorTest

In [11]:
sep_test = SeparatorTest(comp_KRSNL, [210, 5.5, 1.6], [110, 25, 25], 300, 110)

In [12]:
sep_test.calculate()

2026-06-14 20:53:11,470 | _src.PhaseDiagram.new_methodv2 | INFO | Инициализация: T=383.14 K, P_min=0.0100, P_max=1000.0000
2026-06-14 20:53:11,471 | _src.PhaseDiagram.new_methodv2 | INFO | Запуск поиска давления насыщения
2026-06-14 20:53:11,471 | _src.PhaseDiagram.new_methodv2 | INFO | 🔍 Поиск bracket: start=[0.010, 1000.000]
2026-06-14 20:53:11,570 | _src.PhaseDiagram.new_methodv2 | INFO | Диапазон поиска: unstable=184.5785 bar -> stable=185.5550 bar
2026-06-14 20:53:11,737 | _src.PhaseDiagram.new_methodv2 | INFO | Интервал схлопнулся на итерации 12. Возврат P=185.0415 bar


[FlashResult(pressure=300, temperature=383.14, vapor=PhaseState(mole_fraction=0.0, composition={'N2': 0.0032799672003279963, 'CO2': 0.014229857701422984, 'C1': 0.3934560654393456, 'C2': 0.10477895221047788, 'C3': 0.08000919990800091, 'iC4': 0.010309896901030988, 'nC4': 0.033139668603313965, 'iC5': 0.00961990380096199, 'nC5': 0.01888981110188898, 'C6': 0.023539764602353977, 'C7': 0.03264967350326496, 'C8': 0.03346966530334696, 'C9': 0.027859721402785968, 'C10': 0.023979760202397976, 'C11': 0.01967980320196798, 'C12': 0.01625983740162598, 'C13': 0.016019839801601984, 'C14': 0.014179858201417985, 'C15': 0.011739882601173986, 'C16': 0.010029899701002988, 'C17': 0.00868991310086899, 'C18': 0.00812991870081299, 'C19': 0.007559924400755992, 'C20': 0.0066999330006699935, 'C21': 0.0052699473005269944, 'C22': 0.0051299487005129945, 'C23': 0.004369956300436995, 'C24': 0.003959960400395996, 'C25': 0.004059959400405996, 'C26': 0.0032799672003279963, 'C27': 0.003149968500314997, 'C28': 0.00265997340

In [13]:
sep_test._dle_df

,pressure,temperature,is_two_phase,vapor_mole_frac,liquid_mole_frac,vapor_composition,liquid_composition,vapor_density,liquid_density,vapor_molar_density,...,vapor_molar_volume,liquid_molar_volume,vapor_molecular_ weight,liquid_molecular_ weight,vapor_viscosity,liquid_viscosity,vapor_z,liquid_z,Bo,Rs
0,300.000000,383.14,False,0.000000,1.000000,"{'N2': 0.0032799672003279963, 'CO2': 0.0142298...","{'N2': 0.0032799672003279963, 'CO2': 0.0142298...",NaN,0.652477,NaN,...,NaN,139.016422,NaN,90.705065,NaN,0.050051,NaN,1.309872,1.436088,146.864223
1,185.041525,383.14,True,0.000024,0.999976,"{'N2': 0.00918689651221849, 'CO2': 0.016795311...","{'N2': 0.003279827123635907, 'CO2': 0.01422979...",0.169020,0.630973,0.007054,...,141.758225,143.756811,23.959928,90.706648,0.022835,0.048831,0.823870,0.835486,1.485022,146.864223
2,210.000000,383.14,False,0.000000,1.000000,"{'N2': 0.003279827123635907, 'CO2': 0.01422979...","{'N2': 0.003279827123635907, 'CO2': 0.01422979...",NaN,0.635509,NaN,...,NaN,142.730605,NaN,90.706648,NaN,0.049085,NaN,0.941408,1.474422,146.864223
3,5.500000,298.14,True,0.575959,0.424041,"{'N2': 0.005651553121443227, 'CO2': 0.02260976...","{'N2': 5.8405963884851886e-05, 'CO2': 0.002847...",0.005450,0.777692,0.000228,...,4386.771704,233.302308,23.907445,181.437350,0.014022,0.053435,0.973837,0.051792,1.021955,5.737416
4,1.600000,298.14,True,0.055219,0.944781,"{'N2': 0.0010059409071139804, 'CO2': 0.0313526...","{'N2': 3.025830994423974e-06, 'CO2': 0.0011815...",0.002266,0.781878,0.000066,...,15240.802593,243.034769,34.530247,190.023561,0.013467,0.052944,0.984253,0.015695,1.005802,0.000000
5,1.013250,293.14,True,0.010917,0.989083,"{'N2': 0.00023596232484713098, 'CO2': 0.032171...","{'N2': 4.5487494001869166e-07, 'CO2': 0.000839...",0.001650,0.784646,0.000042,...,23719.944555,244.299843,39.141745,191.688867,0.013139,0.052898,0.986630,0.010162,1.000000,0.000000


## Работа через интерфейс  Model

In [46]:
from _src.CompositionalModel.CompositionalModelV2 import CompositionalModel
from _src.Utils.Conditions import Conditions

In [48]:
model = CompositionalModel(comp_KRSNL)

In [49]:
conds = Conditions(100, 383)

In [ ]:
model.flash(110,373)

In [ ]:
model.flash(111, 373)

In [ ]:
model.result_store_object._results[0]

In [ ]:
model.result_store_object.get_by_module('Flash')

In [ ]:
from _src.Utils.Export2 import ModelJSONDB

In [ ]:
model_jsn = ModelJSONDB()

In [ ]:
model_jsn.export('PRRZLM_MDT_TEST', 'BASE', composition.composition, composition.composition_data,
                 composition.eos_name, None, field='PRRZLM')

In [ ]:
model_jsn.save()

## Новый переписанный модуль по расчету Psat

In [40]:
from _src.PhaseDiagram.new_methodv2 import SaturationPressure

In [ ]:
sat_p = SaturationPressure(composition, t_K= 110+273.15, p_max_bar=600)

In [ ]:
sat_p.sp_convergence_loop()

In [ ]:
sat_p.p_i

In [ ]:
import numpy as np

t_arr = np.linspace(273.15, 550+273.15, 20)

for t in t_arr:
    sat_p = SaturationPressure(composition=composition,
                               t_K=t, p_max_bar= 500)
    res_sp = sat_p.sp_convergence_loop()
    res_dp = sat_p.dp_convergence_loop()

    print('====')
    print(f'RESULT: PSAT:{res_sp}, PDEW:{res_dp}, T:{t-273.15}')
    print('====')


## Давление насыщения, давление конденсации и фазовая

In [14]:
from _src.PhaseDiagram.BubblePointPressure import BubblePointCalculator
from _src.PhaseDiagram.DewPressure import DewPointCalculator
from _src.PhaseDiagram.PhaseEnvelope import PhaseDiagramCalculator
from _src.PhaseDiagram.CriticalPoint import CriticalPointCalculator
from _src.PhaseDiagram.PhaseEnvelopeFromStability import PhaseEnvelopeFromStability

### Psat

In [ ]:
p_sat_calc = BubblePointCalculator(comp_KRSNL, 110+273.15)
p_sat_calc.calculate()

### Pdew

In [ ]:
p_dew_calc = DewPointCalculator(comp_KRSNL, 330 + 273.15, P_guess= 260, dew_point_type = '')
p_dew_calc.calculate()

### EnvelopeFromStability

In [27]:
phase_env_from_stab = PhaseEnvelopeFromStability(comp_KRSNL, max_pressure=250, max_temperature=510, pressure_points=100, temperature_points=100)

In [28]:
phase_env_from_stab.run_parallel()

In [ ]:
phase_env_from_stab.plot()

### Envelope

In [ ]:
phase_envelope = PhaseDiagramCalculator(comp_KRSNL, 273.15, 550 + 273.15)
phase_envelope.calculate_phase_envelope()

In [ ]:
phase_envelope.plot_phase_diagram()

#### From VBA

In [49]:
import numpy as np
import logging
from typing import Dict, Optional
from _src.Composition.CompositionV2 import Composition
from _src.PhaseStability.TwoPhaseStabilityTestV3 import TwoPhaseStabilityTest

logger = logging.getLogger(__name__)


class PhaseDiagramCalculator:
    """
    Расчет фазовой диаграммы методом бисекции (как в VBA).
    """
    
    def __init__(self, composition: Composition, T_min: float, T_max: float, dt: float = 5.0):
        self.composition = composition
        self.T_min = T_min
        self.T_max = T_max
        self.dt = dt
        
        self.bubble_points = {'T': [], 'P': []}
        self.dew_points = {'T': [], 'P': []}
    
    def _stability_test(self, P: float, T: float) -> Dict:
        """
        Тест стабильности Michelsen (как в VBA).
        Возвращает: stable, Sv, Sl, K_v, K_l
        """
        stab = TwoPhaseStabilityTest(self.composition, P, T)
        stab.calculate_phase_stability()
        
        return {
            'stable': stab.stable,
            'S_v': stab.S_v,
            'S_l': stab.S_l,
            'K_v': stab.k_values_vapour,
            'K_l': stab.k_values_liquid,
            'K_flash': stab.k_values_for_flash
        }
    
    def _find_saturation_pressure(self, T: float, P_prev: float) -> Optional[Dict]:
        """
        Поиск давления насыщения методом бисекции (как в VBA).
        """
        self.composition.T = T
        
        # Начальные границы
        P_max = 2.0 * P_prev if P_prev > 0 else 300.0
        P_min = 0.0
        P = P_prev if P_prev > 0 else 100.0
        
        # Тест стабильности при текущем P
        stab = self._stability_test(P, T)
        
        if stab['stable']:
            # Система стабильна — нужно найти давление, где она становится нестабильной
            # Идем вниз (bubble) или вверх (dew)
            logger.info(f"T={T-273.15:.1f}°C: Stable at P={P:.2f} bar, searching...")
            
            # Пробуем найти нестабильную область
            for _ in range(20):
                P_test = P * 0.5
                stab_test = self._stability_test(P_test, T)
                
                if not stab_test['stable']:
                    P_max = P
                    P_min = P_test
                    break
                
                P = P_test
            else:
                return None  # Не нашли нестабильную область
        
        # Бисекция
        for _ in range(50):
            if P_max - P_min < 1e-6:
                break
            
            P = (P_max + P_min) / 2.0
            stab = self._stability_test(P, T)
            
            if stab['stable']:
                P_max = P
            else:
                P_min = P
        
        # Определяем тип точки насыщения
        if stab['S_v'] > 1.0 and stab['S_v'] > stab['S_l']:
            # Bubble point
            return {'type': 'bubble', 'P': P, 'S_v': stab['S_v'], 'S_l': stab['S_l']}
        elif stab['S_l'] > 1.0 and stab['S_l'] > stab['S_v']:
            # Dew point
            return {'type': 'dew', 'P': P, 'S_v': stab['S_v'], 'S_l': stab['S_l']}
        else:
            return None
    
    def calculate(self) -> Dict:
        """
        Основной расчет фазовой диаграммы.
        """
        logger.info(f"Starting phase diagram calculation: T=[{self.T_min-273.15:.1f}, {self.T_max-273.15:.1f}]°C")
        
        P_prev = 100.0  # Начальное давление
        
        T = self.T_max
        while T >= self.T_min:
            logger.info(f"\n{'='*60}")
            logger.info(f"T = {T-273.15:.2f}°C")
            
            result = self._find_saturation_pressure(T, P_prev)
            
            if result is not None:
                if result['type'] == 'bubble':
                    self.bubble_points['T'].append(T)
                    self.bubble_points['P'].append(result['P'])
                    logger.info(f"  Bubble point: P = {result['P']:.4f} bar")
                else:
                    self.dew_points['T'].append(T)
                    self.dew_points['P'].append(result['P'])
                    logger.info(f"  Dew point: P = {result['P']:.4f} bar")
                
                P_prev = result['P']
            else:
                logger.info(f"  No saturation point found")
            
            T -= self.dt
        
        return {
            'bubble': self.bubble_points,
            'dew': self.dew_points
        }



In [50]:
phase_diag_vba = PhaseDiagramCalculator(comp_KRSNL, T_min=280, T_max=500 + 273.15, dt=20)

In [ ]:
phase_diag_vba.calculate()

In [52]:
import pandas as pd
upp_df = pd.DataFrame.from_dict(phase_diag_vba.bubble_points)
low_df = pd.DataFrame.from_dict(phase_diag_vba.dew_points)

In [ ]:
upp_df

In [ ]:
low_df

In [55]:
import matplotlib.pyplot as plt

In [56]:
full_df = pd.concat([upp_df, low_df], ignore_index=True)


In [ ]:
plt.scatter(full_df['T'], full_df['P'])

### CritPoint

In [ ]:
crit_p = CriticalPointCalculator(comp_KRSNL)
crit_p.calculate()

#### Parallel

In [ ]:
crit_p_par = CriticalPointCalculatorParallel(comp_KRSNL)
crit_p_par.calculate()

### Версия Дениса по расчету Pdew - нужно передавать K-факторы отдельно (где я их должен взять :/)

In [6]:
from _src.PhaseDiagram.SaturationPressure_den_v import DewPointCalculator

In [12]:
sat_p_den = DewPointCalculator(comp_KRSNL, T=350 + 273.15)

In [ ]:
sat_p_den.calculate()

## Critical Point

In [ ]:
# _src/PVT/CriticalPointCalculatorV1.py
import numpy as np
from _src.Composition.CompositionV2 import Composition
from _src.EOS.BrusilovskiyEOSV2 import BrusilovskiyEOS
from _src.PhaseStability.TwoPhaseStabilityTestV3 import TwoPhaseStabilityTest
from _src.Utils.Errors import StopIterationError

class CriticalPointCalculator:
    """
    Расчет критической точки многокомпонентной смеси методом сканирования (T, P) сетки.
    Использует ваш TwoPhaseStabilityTest для определения перехода двухфазной области в однофазную.
    
    Критерий критической точки:
      - S_v -> 1.0 и S_l -> 1.0 (фазовые фракции стремятся к единице)
      - K-values сходятся к тривиальному решению (K_i -> 1.0)
    """
    def __init__(
        self, 
        composition: Composition, 
        t_range: tuple = None, 
        p_range: tuple = None, 
        t_step: float = 5.0, 
        p_step: float = 1.0, 
        tol_metric: float = 1e-6
    ):
        self.comp = composition
        self.t_step = t_step
        self.p_step = p_step
        self.tol_metric = tol_metric

        # Автоматический расчет диапазонов, если не заданы
        tc_vals = np.array([composition.composition_data['critical_temperature'][c] for c in composition.composition])
        pc_vals = np.array([composition.composition_data['critical_pressure'][c] for c in composition.composition])

        self.t_min, self.t_max = t_range or (tc_vals.min() * 0.8, tc_vals.max() * 1.15)
        self.p_min, self.p_max = p_range or (1.0, pc_vals.max() * 1.3)

        self._best_T = None
        self._best_P = None
        self._best_stab = None
        self._min_metric = float('inf')
        self._history = []

    def calculate(self) -> dict:
        """
        Запускает поиск по сетке (T, P).
        Возвращает словарь с результатами или None, если поиск не дал сходимости.
        """
        t_grid = np.arange(self.t_min, self.t_max, self.t_step)
        p_grid = np.arange(self.p_min, self.p_max, self.p_step)

        for T in t_grid:
            for P in p_grid:
                try:
                    self.comp.T = T
                    stab = TwoPhaseStabilityTest(self.comp, P, T)
                    stab.calculate_phase_stability()

                    # Метрика близости к критической точке
                    s_v = stab.S_v if stab.S_v is not None else 100.0
                    s_l = stab.S_l if stab.S_l is not None else 100.0
                    
                    # Отклонение S от 1.0
                    metric = abs(s_v - 1.0) + abs(s_l - 1.0)

                    # Бонус за сходимость в тривиальное решение (K -> 1)
                    if stab.convergence_trivial_solution_v and stab.convergence_trivial_solution_l:
                        metric *= 0.01

                    if metric < self._min_metric:
                        self._min_metric = metric
                        self._best_T, self._best_P, self._best_stab = T, P, stab

                        # Ранний выход, если достигнута высокая точность
                        if self._min_metric < self.tol_metric:
                            break

                except (StopIterationError, ZeroDivisionError, ValueError, RuntimeError):
                    # Игнорируем точки, где EOS или тест стабильности не сходятся
                    continue
            if self._min_metric < self.tol_metric:
                break

        if self._best_stab is None:
            return None

        return {
            'T_crit': float(self._best_T),
            'P_crit': float(self._best_P),
            'S_v': float(self._best_stab.S_v),
            'S_l': float(self._best_stab.S_l),
            'K_v': self._best_stab.k_values_vapour,
            'K_l': self._best_stab.k_values_liquid,
            'metric': float(self._min_metric),
            'is_trivial_converged': bool(
                self._best_stab.convergence_trivial_solution_v and 
                self._best_stab.convergence_trivial_solution_l
            )
        }

    def refine_with_newton(self, T0: float, P0: float, max_iter: int = 20):
        """
        ЗАГОТОВКА НА БУДУЩЕЕ: уточнение точки методом Ньютона/оптимизации.
        После тестов сетки здесь можно подключить scipy.optimize.root или 
        аналитический расчет производных из BrusilovskiyEOSV2.
        """
        pass  # Реализуем на следующем этапе по вашему запросу

In [ ]:
crit_point_obj = CriticalPointCalculator(composition)

In [ ]:
crit_point_obj.calculate()

### Через параллельный счет

In [ ]:
import numpy as np
from typing import Optional, Dict, Any, Tuple
from joblib import Parallel, delayed
from _src.Composition.CompositionV2 import Composition
from _src.PhaseStability.TwoPhaseStabilityTestV3 import TwoPhaseStabilityTest


def _evaluate_grid_point(composition: Composition, t: float, p: float) -> Optional[Dict[str, Any]]:
    """
    Вычисляет метрику близости к критической точке для заданных (T, P).
    Вынесена на уровень модуля для корректной сериализации в joblib.
    """
    try:
        comp = composition.new_composition(composition.composition, deep_copy=True)
        comp.T = t
        stab = TwoPhaseStabilityTest(comp, p, t)
        stab.calculate_phase_stability()

        s_v = stab.S_v
        s_l = stab.S_l

        if s_v is None or s_l is None:
            return None

        # 1. Базовая метрика: суммарное отклонение фазовых фракций от 1.0
        metric = abs(s_v - 1.0) + abs(s_l - 1.0)

        # 2. Бонус за сходимость в тривиальное решение (K_i -> 1)
        if stab.convergence_trivial_solution_v and stab.convergence_trivial_solution_l:
            metric *= 0.01

        # 3. Штраф за отсутствие сходимости хотя бы одного цикла (v/l)
        conv_v = stab.convergence_v or stab.convergence_trivial_solution_v
        conv_l = stab.convergence_l or stab.convergence_trivial_solution_l
        if not (conv_v and conv_l):
            metric += 10.0

        return {
            'T': float(t) - 273.15,
            'P': float(p),
            'metric': float(metric),
            'S_v': float(s_v),
            'S_l': float(s_l),
            'K_v': stab.k_values_vapour,
            'K_l': stab.k_values_liquid,
            'is_trivial': bool(stab.convergence_trivial_solution_v and stab.convergence_trivial_solution_l),
            'stable': stab.stable
        }
    except Exception:
        # Игнорируем точки, где EOS не сходится, возникает деление на ноль или StopIterationError
        return None




class CriticalPointCalculatorParrallel:
    """
    Расчет критической точки многокомпонентной смеси методом сканирования сетки (T, P).
    Использует joblib для параллельного выполнения независимых расчетов стабильности.
    Автоматически определяет границы поиска по Tc и Pc компонентов, если они не заданы явно.
    """
    def __init__(
        self,
        composition: Composition,
        t_range: Optional[Tuple[float, float]] = None,
        p_range: Optional[Tuple[float, float]] = None,
        t_step: float = 5.0,
        p_step: float = 5.0,
        n_jobs: int = -1,
        backend: str = 'loky'
    ):
        self.composition = composition
        self.t_step = t_step
        self.p_step = p_step
        self.n_jobs = n_jobs
        self.backend = backend

        comp_data = composition.composition_data
        components = tuple(composition.composition.keys())

        # Проверка наличия критических параметров
        if not comp_data.get('critical_temperature') or not comp_data.get('critical_pressure'):
            raise RuntimeError(
                "Необходимо предварительно вызвать composition.evaluate_composition_data(), "
                "чтобы заполнить critical_temperature и critical_pressure."
            )

        # Автоматическая генерация границ
        tc_vals = np.fromiter(
            (comp_data['critical_temperature'][c] for c in components), dtype=np.float64
        )
        pc_vals = np.fromiter(
            (comp_data['critical_pressure'][c] for c in components), dtype=np.float64
        )

        self.t_min, self.t_max = t_range or (float(tc_vals.min() * 0.5), float(tc_vals.max() * 1.5))
        self.p_min, self.p_max = p_range or (1.0, float(pc_vals.max() * 3))

    def calculate(self) -> Optional[Dict[str, Any]]:
        """
        Запускает параллельный поиск по сетке и возвращает параметры точки 
        с минимальной метрикой близости к критическому состоянию.
        """
        # Генерация сетки (с учётом погрешности float для включения верхней границы)
        t_grid = np.arange(self.t_min, self.t_max + self.t_step * 0.5, self.t_step)
        p_grid = np.arange(self.p_min, self.p_max + self.p_step * 0.5, self.p_step)
        points = [(t, p) for t in t_grid for p in p_grid]

        # Параллельное выполнение
        with Parallel(n_jobs=self.n_jobs, backend=self.backend) as parallel:
            results = parallel(
                delayed(_evaluate_grid_point)(self.composition, t, p)
                for t, p in points
            )

        # Фильтрация успешных расчетов
        valid_results = [r for r in results if r is not None]
        if not valid_results:
            return None

        # Выбор точки с наименьшей метрикой
        return min(valid_results, key=lambda x: x['metric'])


In [ ]:
crit_point_obj_par = CriticalPointCalculatorParrallel(comp_KRSNL)
crit_point_obj_par.calculate()

# CCE в параллель - неэффективно

In [ ]:
from _src.Experiments.CCE2 import CCE

In [ ]:
cce = CCE(comp_KRSNL, [500, 400, 390, 370, 360, 350, 340, 330, 320, 310, 300, 290, 280, 270, 250, 240,230,220, 210, 200, 190], 110)

In [ ]:
cce.calculate()

In [ ]:
cce.calculate_parallel()